In [7]:
# Imports
import pandas as pd
from datetime import date

from google.cloud import bigquery

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

import meteostat as ms

import time
import requests

In [2]:
# ------------------------------------------------------------------
# Configuration
# ------------------------------------------------------------------

PROJECT_ID = "pacey32-agency"

client = bigquery.Client(project=PROJECT_ID)

geolocator = Nominatim(
    user_agent="pacey32-hockey-climate",
    timeout=10
)

geocode = RateLimiter(
    geolocator.geocode,
    min_delay_seconds=1.1
)

In [3]:
# ------------------------------------------------------------------
# Read cities from BigQuery
# ------------------------------------------------------------------

sql = """
SELECT DISTINCT venueLocation
FROM `pacey32-agency.Team.TeamList`
ORDER BY venueLocation
"""

cities = client.query(sql).to_dataframe()

In [5]:
# ------------------------------------------------------------------
# Geocode
# ------------------------------------------------------------------

def get_lat_lon(city):

    location = geocode(city)

    if location:
        return pd.Series(
            [location.latitude, location.longitude]
        )

    return pd.Series([None, None])


cities[["latitude", "longitude"]] = (
    cities["venueLocation"]
    .apply(get_lat_lon)
)

In [6]:
cities.head()

,venueLocation,latitude,longitude
0,Anaheim,33.834752,-117.911732
1,Boston,42.358834,-71.057830
2,Buffalo,42.886416,-78.878149
3,Calgary,51.045606,-114.057541
4,Chicago,41.875562,-87.624421


In [11]:
city_limit = cities.head(2)

In [12]:
# ------------------------------------------------------------------
# Climate API configuration
# ------------------------------------------------------------------

import time
import requests

CLIMATE_START = "2021-01-01"
CLIMATE_END = "2050-12-31"
CLIMATE_MODEL = "MRI_AGCM3_2_S"
API_URL = "https://climate-api.open-meteo.com/v1/climate"

daily_fields = [
    "temperature_2m_mean",
    "temperature_2m_min",
    "temperature_2m_max",
    "precipitation_sum",
    "snowfall_sum",
    "cloud_cover_mean",
    "shortwave_radiation_sum"
]

climate_results = []

for _, row in city_limit.iterrows():

    city = row["venueLocation"]
    lat = row["latitude"]
    lon = row["longitude"]

    if pd.isna(lat) or pd.isna(lon):
        print(f"Skipping {city}: no coordinates")
        continue

    print(f"Getting climate data for {city}")

    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": CLIMATE_START,
        "end_date": CLIMATE_END,
        "models": CLIMATE_MODEL,
        "daily": ",".join(daily_fields)
    }

    response = requests.get(
        API_URL,
        params=params,
        timeout=120
    )

    while response.status_code == 429:
        print("Rate limited. Waiting 60 seconds.")
        time.sleep(60)

        response = requests.get(
            API_URL,
            params=params,
            timeout=120
        )

    response.raise_for_status()

    data = response.json()

    df = pd.DataFrame(data["daily"])

    df["time"] = pd.to_datetime(df["time"])
    df["Year"] = df["time"].dt.year
    df["Month"] = df["time"].dt.month

    # First calculate each individual year/month
    year_month = (
        df
        .groupby(
            ["Year", "Month"],
            as_index=False
        )
        .agg(
            AvgTemp=("temperature_2m_mean", "mean"),
            MinTemp=("temperature_2m_min", "mean"),
            MaxTemp=("temperature_2m_max", "mean"),
            RainMM=("precipitation_sum", "sum"),
            Snowfall=("snowfall_sum", "sum"),
            CloudCover=("cloud_cover_mean", "mean"),
            SolarRadiation=("shortwave_radiation_sum", "mean")
        )
    )

    # Then average each calendar month across 2021–2050
    out = (
        year_month
        .groupby(
            "Month",
            as_index=False
        )
        .agg(
            AvgTemp=("AvgTemp", "mean"),
            MinTemp=("MinTemp", "mean"),
            MaxTemp=("MaxTemp", "mean"),
            RainMM=("RainMM", "mean"),
            Snowfall=("Snowfall", "mean"),
            CloudCover=("CloudCover", "mean"),
            SolarRadiation=("SolarRadiation", "mean")
        )
    )

    out["venueLocation"] = city
    out["latitude"] = lat
    out["longitude"] = lon
    out["climate_start"] = CLIMATE_START
    out["climate_end"] = CLIMATE_END
    out["climate_model"] = CLIMATE_MODEL
    out["api_url"] = API_URL
    out["last_updated"] = pd.Timestamp.now(tz="UTC")

    climate_results.append(out)

    time.sleep(1)

climate_df = pd.concat(
    climate_results,
    ignore_index=True
)

climate_df = climate_df[
    [
        "venueLocation",
        "latitude",
        "longitude",
        "Month",
        "AvgTemp",
        "MinTemp",
        "MaxTemp",
        "RainMM",
        "Snowfall",
        "CloudCover",
        "SolarRadiation",
        "climate_start",
        "climate_end",
        "climate_model",
        "api_url",
        "last_updated"
    ]
]

Getting climate data for Anaheim
Getting climate data for Boston


In [13]:
climate_df.head()

,venueLocation,latitude,longitude,Month,AvgTemp,MinTemp,MaxTemp,RainMM,Snowfall,CloudCover,SolarRadiation,climate_start,climate_end,climate_model,api_url,last_updated
0,Anaheim,33.834752,-117.911732,1,12.416774,7.512688,18.683226,54.101000,0.0,32.859140,11.382097,2021-01-01,2050-12-31,MRI_AGCM3_2_S,https://climate-api.open-meteo.com/v1/climate,2026-07-30 07:57:14.447206+00:00
1,Anaheim,33.834752,-117.911732,2,13.938883,8.938736,20.295878,34.641333,0.0,31.415517,14.963165,2021-01-01,2050-12-31,MRI_AGCM3_2_S,https://climate-api.open-meteo.com/v1/climate,2026-07-30 07:57:14.447206+00:00
2,Anaheim,33.834752,-117.911732,3,14.391720,9.270753,20.890645,30.610667,0.0,31.311828,19.676140,2021-01-01,2050-12-31,MRI_AGCM3_2_S,https://climate-api.open-meteo.com/v1/climate,2026-07-30 07:57:14.447206+00:00
3,Anaheim,33.834752,-117.911732,4,16.218778,11.237444,22.559444,19.417333,0.0,31.775556,23.981322,2021-01-01,2050-12-31,MRI_AGCM3_2_S,https://climate-api.open-meteo.com/v1/climate,2026-07-30 07:57:14.447206+00:00
4,Anaheim,33.834752,-117.911732,5,18.674731,13.748495,24.678172,5.259000,0.0,28.260215,27.036398,2021-01-01,2050-12-31,MRI_AGCM3_2_S,https://climate-api.open-meteo.com/v1/climate,2026-07-30 07:57:14.447206+00:00
